In [0]:
-- 步骤1: 创建临时视图读取Bronze数据
-- 获取 ADF 传递的参数用新方法
-- DECLARE tdspath STRING DEFAULT getArgument('p_tdspath', 'gmall/order_info/default');
CREATE OR REPLACE TEMPORARY VIEW bronze_user_info_raw AS
SELECT 
    *,
    _metadata.file_path as source_file,
    current_timestamp() as load_timestamp
FROM read_files('abfss://bronze@sahyivy.dfs.core.windows.net/'||:p_tdspath||'/*.parquet');

-- 步骤2: 合并新数据到Silver表（UPSERT）
MERGE INTO silver_user_info AS target
USING bronze_user_info_raw AS source
ON target.id = source.id
    WHEN MATCHED THEN
        UPDATE SET                
                target.login_name             =source.login_name            ,
                target.nick_name              =source.nick_name             ,
                target.passwd                 =source.passwd                ,
                target.name                   =source.name                  ,
                target.phone_num              =source.phone_num             ,
                target.email                  =source.email                 ,
                target.head_img               =source.head_img              ,
                target.user_level             =source.user_level            ,
                target.birthday               =source.birthday              ,
                target.gender                 =source.gender                ,
                target.create_time            =source.create_time           ,
                target.operate_time           =source.operate_time          ,
                target.status                 =source.status                ,
                target.source_file            =source.source_file           ,
                target.load_timestamp         =source.load_timestamp        ,
                target.update_timestamp = current_timestamp()
    WHEN NOT MATCHED THEN
        INSERT (id          ,
                login_name            ,
                nick_name             ,
                passwd                ,
                name                  ,
                phone_num             ,
                email                 ,
                head_img              ,
                user_level            ,
                birthday              ,
                gender                ,
                create_time           ,
                operate_time          ,
                status                ,
                source_file           ,
                load_timestamp        ,
                update_timestamp
                )
        VALUES (source.id          ,
                source.login_name            ,
                source.nick_name             ,
                source.passwd                ,
                source.name                  ,
                source.phone_num             ,
                source.email                 ,
                source.head_img              ,
                source.user_level            ,
                source.birthday              ,
                source.gender                ,
                source.create_time           ,
                source.operate_time          ,
                source.status                ,
                source.source_file           , 
                source.load_timestamp        ,
                current_timestamp()
                );